In [ ]:
import numpy as np
from scipy.linalg import cholesky, det
from datetime import datetime
import time


def BINGO(data, state, parameters):
    stats = {"start_time": datetime.now()}

    if match_check(data["Tsam"], data["ts"]):
        raise ValueError("Sampling times are not consistent with the time series data!")

    # Set the state of the sampler
    q       = state["q"].copy()
    gamma   = state["gamma"].copy()
    r       = state["r"].copy()
    xs_old  = state["xs"].copy()
    Pold    = state["P"].copy()
    betsold = state["bets"].copy()
    Jold    = state["J"].copy()
    Sold    = state["S"].copy()
    psiold  = state["psi"].copy()
    ma      = state["ma"].copy()
    mb      = state["mb"].copy()

    # Heuristic parameter to speed up topology sampling (default = 1)
    Theur = parameters.get("Theur", 1)

    # Reform data
    Tsam = data["Tsam"]
    ts   = data["ts"]
    u    = np.empty((0, 0))

    if "input" in data:
        if match_check(data["input"], data["ts"]):
            raise ValueError("Input size is not consistent with the time series data!")
        input_data = data["input"]
        u = input_data[0][:, :-1]

    # TS data in one matrix y; Ser tracks indices for different experiments
    y = ts[0].copy()
    n_ser = len(ts)
    Ser = np.zeros((4, n_ser), dtype=int)
    Ser[0, 0] = 0                       # 0-indexed start
    Ser[1, 0] = y.shape[1] - 1         # 0-indexed end (inclusive)

    for j in range(1, n_ser):
        y = np.hstack([y, ts[j]])
        Ser[0, j] = Ser[1, j-1] + 1
        Ser[1, j] = Ser[1, j-1] + ts[j].shape[1]
        if "input" in data:
            u = np.hstack([u, input_data[j][:, :-1]])

    # Check dimensions
    n    = y.shape[0]
    n_in = u.shape[0] if u.size > 0 else 0
    nstep = (xs_old.shape[1] - n_ser) // (y.shape[1] - n_ser)
    M    = psiold.shape[1]

    all_data = np.vstack([y, u]) if u.size > 0 else y
    rany = np.column_stack([all_data.min(axis=1), all_data.max(axis=1)])

    # Which variables are included per experiment (knockout handling)
    Incl = np.ones((n, n_ser))
    if "ko" in data:
        for jser in range(n_ser):
            Incl[data["ko"][jser], jser] = 0

    gene_list     = np.where(Incl.sum(axis=1) > 0)[0]
    excluded_genes = np.setdiff1d(np.arange(n), gene_list)

    # Prior probability for link existence (default = 1/n)
    if "link_pr" in parameters:
        log_link_pr = np.log(parameters["link_pr"])
    else:
        log_link_pr = -np.log(n) * np.ones((n, n + n_in))
    if log_link_pr.size == 1:
        log_link_pr = log_link_pr * np.ones((n, n + n_in))

    # Process prior network information
    S_aux = np.zeros((n, n + n_in))
    if "sure" in data:
        sure = data["sure"]
        if sure.shape[1] > n + 0.5:
            S_aux = sure.copy()
        else:
            S_aux[:n, :n] = sure
    Sold = np.maximum(Sold, S_aux)
    Sold = np.minimum(Sold, 1 + S_aux)
    S_aux = np.ones((n, n + n_in)) - np.abs(S_aux)
    S_aux[:, excluded_genes] = 0
    S_aux[excluded_genes, :] = 0
    Sold[:, excluded_genes] = 0
    Sold[excluded_genes, :] = 0

    # Rows 3-4 of Ser: indices in the finer grid
    Ser[2, 0] = 0
    Ser[3, 0] = nstep * (Ser[1, 0] - Ser[0, 0])   # inclusive end on fine grid
    for jser in range(1, n_ser):
        Ser[2, jser] = Ser[3, jser-1] + 1
        Ser[3, jser] = Ser[3, jser-1] + nstep * (Ser[1, jser] - Ser[0, jser])

    # Expand inputs to fine grid
    if u.size > 0:
        u_expanded = []
        for j in range(u.shape[1]):
            u_expanded.append(np.tile(u[:, j:j+1], (1, nstep)))
        u = np.hstack(u_expanded)

    # Missing measurements
    nomiss = np.ones_like(y)
    if "missing" in data:
        if len(data["missing"]) != len(data["ts"]):
            raise ValueError("Missing measurements indices are not consistent with the time series data!")
        for i in range(n):
            for jser in range(n_ser):
                miss_jser = data["missing"][jser]
                if miss_jser.shape[0] > 0:
                    miss_rows = np.where(np.abs(miss_jser[:, 0] - i) < 0.5)[0]
                    if len(miss_rows) > 0:
                        cols = miss_jser[miss_rows, 1].astype(int) + Ser[0, jser]
                        nomiss[i, cols] = 0

    # Fine-grid indices
    derind_full = np.arange(nstep * (Ser[1, 0] - Ser[0, 0]))
    yind        = np.arange(0, Ser[1, 0] + 1) * nstep
    d_full      = np.repeat(
        (Tsam[0][1:] - Tsam[0][:-1]) / nstep,
        nstep
    )
    for jser in range(1, n_ser):
        offset      = derind_full[-1] + 2
        new_derind  = np.arange(nstep * (Ser[1, jser] - Ser[0, jser])) + offset
        derind_full = np.concatenate([derind_full, new_derind])
        yind        = np.concatenate([yind, yind[-1] + 1 + np.arange(0, Ser[1, jser] - Ser[0, jser] + 1) * nstep])
        d_full      = np.concatenate([d_full, np.repeat((Tsam[jser][1:] - Tsam[jser][:-1]) / nstep, nstep)])

    # Signal variation
    nry      = np.zeros(n)
    totvar   = np.zeros(n)
    Total_time = np.zeros(n)
    for l in range(n_ser):
        sl, el = Ser[0, l], Ser[1, l]
        dt = Tsam[l][1:] - Tsam[l][:-1]
        nry    += np.sum((y[:, sl+1:el+1] - y[:, sl:el])**2 / dt, axis=1)
        totvar += np.sum(np.abs(y[:, sl+1:el+1] - y[:, sl:el]), axis=1)
        Total_time += Incl[:, l] * (Tsam[l][-1] - Tsam[l][0])
    nry    /= Total_time
    totvar /= Total_time

    # Piecewise-linear embedding matrix Pr
    mm = int(Ser[3, :].max() - Ser[2, :].min()) + 1
    max_y_span = int((Ser[1, :] - Ser[0, :]).max()) + 1
    Pr = np.zeros((mm, max_y_span))
    Pr[:nstep, 0]   = np.arange(nstep, 0, -1) / nstep
    Pr[mm-nstep:, -1] = np.arange(1, nstep+1) / nstep
    for j in range(1, max_y_span - 1):
        Pr[(j-1)*nstep+1:j*nstep+1, j]   = np.arange(1, nstep+1) / nstep
        Pr[j*nstep:j*nstep+nstep, j] = np.arange(nstep, 0, -1) / nstep

    idx = np.arange(1, nstep)
    Pintc = (np.sin(np.outer(idx, idx) / nstep * np.pi)
             / (np.pi * idx[np.newaxis, :]) * 2**0.5)

    # Storage
    Plink    = np.zeros_like(Sold)
    chain    = 0
    acctraj  = 0
    xstore   = np.zeros_like(xs_old)
    acctop   = np.zeros(n)
    acchyp   = np.zeros(n)
    accr     = np.zeros(n)
    yold     = xs_old[:, yind]

    # -----------------------------------------------------------------------
    # Main iterations
    # -----------------------------------------------------------------------
    t0 = time.time()
    time_mark = t0

    for k in range(1, parameters["its"] + 1):

        # ----------------------------------------------------------------
        # Topology sampling
        # ----------------------------------------------------------------
        for i in gene_list:
            S = Sold[i, :].copy()
            top_change = int(np.random.rand() > 0.333)

            inds = np.where(S_aux[i, :] > 0.5)[0]
            topc = int(
                np.random.rand() > 0.5
                and S[inds].sum() > 0.5
                and S[inds].sum() < S_aux[i, :].sum() - 0.5
            )

            # Move type 1: flip one entry
            if top_change and not topc:
                indc = np.random.randint(int(S_aux[i, :].sum()))
                S[inds[indc]] = 1 - S[inds[indc]]

            # Move type 2: swap a 0→1 and a 1→0
            if top_change and topc:
                ind1 = np.where(S[inds] > 0.5)[0]
                ind0 = np.where(S[inds] < 0.5)[0]
                indc01 = ind0[np.random.randint(len(ind0))]
                indc10 = ind1[np.random.randint(len(ind1))]
                S[inds[indc01]] = 1
                S[inds[indc10]] = 0

            # Sample relevance parameters
            bets = ((1 - parameters["ebeta"]**2)**0.5 * betsold[i, :]
                    + parameters["ebeta"] * np.random.randn(n + n_in))
            beta = 0.5 + 0.45 * bets
            p_bets = np.exp(-np.abs(beta)) / np.exp(-(beta - 0.5)**2 / (2 * 0.45**2))
            beta = np.abs(beta)

            # Sample other hyperparameters
            gamma_tr = gamma[i] + parameters["egamma"] * nry[i] * np.random.randn()
            gamma_tr = 1e-4 + abs(gamma_tr - 1e-4)
            matr = ma[i] + parameters["ea"] * np.random.randn()
            matr = 1e-7 + abs(matr - 1e-7)
            mbtr = mb[i] + parameters["eb"] * np.random.randn()
            mbtr = 1e-7 + abs(mbtr - 1e-7)

            # Exclude knockout data for gene i
            d_i = d_full.copy()
            di_full = derind_full.copy()
            if Incl[i, :].sum() < n_ser - 0.5:
                di_full = np.array([], dtype=int)
                d_i     = np.array([])
                for jser in np.where(Incl[i, :] > 0.5)[0]:
                    seg = np.arange(Ser[2, jser], Ser[3, jser]) - jser
                    di_full = np.concatenate([di_full, derind_full[seg]])
                    d_i     = np.concatenate([d_i,     d_full[seg]])
            N = len(di_full)

            # Form covariance matrices
            KM  = np.zeros((M, M))
            KNM = np.zeros((N, M))
            for j in np.where(S[:n] > 0.5)[0]:
                diff_M  = psiold[j, :][np.newaxis, :] - psiold[j, :][:, np.newaxis]
                diff_NM = psiold[j, :][np.newaxis, :] - xs_old[j, di_full][:, np.newaxis]
                KM  += beta[j] * diff_M**2
                KNM += beta[j] * diff_NM**2
            for jj in np.where(S[n:n+n_in] > 0.5)[0]:
                j = n + jj
                diff_M  = psiold[j, :][np.newaxis, :] - psiold[j, :][:, np.newaxis]
                diff_NM = psiold[j, :][np.newaxis, :] - u[jj, :][di_full][:, np.newaxis]
                KM  += beta[j] * diff_M**2
                KNM += beta[j] * diff_NM**2
            KM  = gamma_tr * np.exp(-KM)
            KNM = gamma_tr * np.exp(-KNM)

            KC  = cholesky(KM + (1/q[i]) * (KNM.T @ (d_i[:, None] * KNM)) + 1e-5 * np.eye(M))
            der = ((xs_old[i, di_full+1] - xs_old[i, di_full])
                   - d_i * (mbtr - matr * xs_old[i, di_full])) / q[i]
            ld  = np.linalg.solve(KC.T, KNM.T @ der)

            nrY = sum(
                np.sum((yold[i, Ser[0,l]+1:Ser[1,l]+1] - yold[i, Ser[0,l]:Ser[1,l]])**2
                       / (Tsam[l][1:] - Tsam[l][:-1]))
                for l in range(n_ser)
            )

            J1 = (0.5 * nrY / q[i]
                  - 0.5 * ld @ ld
                  + np.log(np.diag(KC)).sum()
                  - 0.5 * np.log(det(KM + 1e-5 * np.eye(M)))
                  - (mbtr - matr * xs_old[i, di_full]) @ (xs_old[i, di_full+1] - xs_old[i, di_full]) / q[i]
                  + 0.5 / q[i] * np.sum((d_i**0.5 * (mbtr - matr * xs_old[i, di_full]))**2))
            PS = np.sum(S * log_link_pr[i, :]) + np.log(p_bets).sum()

            P_aux_ab = np.exp(0.1 * (ma[i] - matr + 2*mb[i] - 2*mbtr) / totvar[i])
            P_aux_gamma = (gamma_tr / nry[i] * (30 - gamma_tr / nry[i])
                           / (gamma[i] / nry[i] * (30 - gamma[i] / nry[i]))
                           * np.exp(0.2 / nry[i] * (gamma[i] - gamma_tr)))

            if (P_aux_ab * P_aux_gamma
                    * np.exp((PS - Pold[i] + Jold[i] - J1) / Theur) > np.random.rand()):
                Sold[i, :] = S
                Pold[i]    = PS
                Jold[i]    = J1
                betsold[i, :] = bets
                gamma[i]  = gamma_tr
                ma[i]     = matr
                mb[i]     = mbtr
                acctop[i] += top_change
                acchyp[i] += 1

            # Sample measurement noise variance r(i)
            rtr = r[i] + parameters["er"] * np.random.randn()
            rtr = 1e-8 + abs(rtr - 1e-8)
            obs = np.where(nomiss[i, :] > 0.5)[0]
            if ((r[i] / rtr)**(1 + len(obs) / 2)
                    * np.exp(1e-5 / r[i] - 1e-5 / rtr
                             + np.sum((y[i, obs] - yold[i, obs])**2) / 2 * (1/r[i] - 1/rtr))
                    > np.random.rand()):
                r[i]    = rtr
                accr[i] += 1

        # ----------------------------------------------------------------
        # Trajectory sampling
        # ----------------------------------------------------------------
        qtr = q + parameters["eq"] * np.random.randn(*q.shape)
        qtr = 0.5e-5 + np.abs(qtr - 0.5e-5)

        xs   = np.zeros_like(xs_old)
        yhat = np.zeros_like(y)

        for l in range(n_ser):
            sl, el = Ser[0, l], Ser[1, l]
            sf, ef = Ser[2, l], Ser[3, l]
            nt = el - sl       # number of intervals

            if "missing" in data and data["missing"][l].shape[0] > 0.5:
                for i in range(n):
                    Csam = missing_data_sampler(data["missing"][l], Tsam[l], r[i], qtr[i], i)
                    coef = ((1 - parameters["etraj"]**2)**0.5 * nomiss[i, sl:el+1]
                            + (qtr[i]/q[i])**0.5 * (1 - nomiss[i, sl:el+1]))
                    yhat[i, sl:el+1] = (y[i, sl:el+1]
                                        + coef * (yold[i, sl:el+1] - y[i, sl:el+1])
                                        + parameters["etraj"] * np.random.randn(el-sl+1) @ Csam.T)
            else:
                yhat[:, sl:el+1] = (y[:, sl:el+1]
                                    + (1 - parameters["etraj"]**2)**0.5
                                    * (yold[:, sl:el+1] - y[:, sl:el+1])
                                    + parameters["etraj"] * np.diag(r**0.5)
                                    @ np.random.randn(n, el-sl+1))

            scale = np.diag((qtr / q)**0.5)
            Pr_sub = Pr[:ef-sf+1, :nt+1]
            xs[:, sf:ef+1] = (scale @ ((1 - parameters["etraj"]**2)**0.5 * xs_old[:, sf:ef+1])
                              + (yhat[:, sl:el+1]
                                 - scale @ ((1 - parameters["etraj"]**2)**0.5 * yold[:, sl:el+1]))
                              @ Pr_sub.T)

            # Build Brownian bridge noise, one column block per interval
            # Each interval contributes nstep fine steps.
            # Final shape needed: (n, nt*nstep)
            cols = []
            for iv in range(nt):
                d_iv = d_full[sf + iv * nstep]          # scalar dt for this interval
                raw  = np.random.randn(nstep - 1, n)    # (nstep-1, n)
                seg  = Pintc @ raw                       # (nstep-1, n)
                seg  = np.vstack([seg, np.zeros((1, n))])  # (nstep, n)
                seg  = (parameters["etraj"]
                        * np.diag(qtr**0.5)
                        @ ((nstep * d_iv)**0.5 * seg.T))    # (n, nstep)
                cols.append(seg)
            noise_block = np.hstack(cols)               # (n, nt*nstep)
            xs[:, sf+1:sf+1+nt*nstep] += noise_block

        # Pseudoinput proposal
        psin = psiold + 0.025 * np.random.randn(*psiold.shape)
        psin = np.minimum(psin, 2 * rany[:, 1:2] - psin)
        psin = np.maximum(psin, 2 * rany[:, 0:1] - psin)

        J1 = np.zeros(n)

        for i in gene_list:
            d_i     = d_full.copy()
            di_full = derind_full.copy()
            if Incl[i, :].sum() < n_ser - 0.5:
                di_full = np.array([], dtype=int)
                d_i     = np.array([])
                for jser in np.where(Incl[i, :] > 0.5)[0]:
                    seg = np.arange(Ser[2, jser], Ser[3, jser]) - jser
                    di_full = np.concatenate([di_full, derind_full[seg]])
                    d_i     = np.concatenate([d_i,     d_full[seg]])
            N = len(di_full)

            beta = np.abs(0.5 + 0.45 * betsold)
            KM  = np.zeros((M, M))
            KNM = np.zeros((N, M))
            for j in np.where(Sold[i, :n] > 0.5)[0]:
                diff_M  = psin[j, :][np.newaxis, :] - psin[j, :][:, np.newaxis]
                diff_NM = psin[j, :][np.newaxis, :] - xs[j, di_full][:, np.newaxis]
                KM  += beta[i, j] * diff_M**2
                KNM += beta[i, j] * diff_NM**2
            for jj in np.where(Sold[i, n:n+n_in] > 0.5)[0]:
                j = n + jj
                diff_M  = psin[j, :][np.newaxis, :] - psin[j, :][:, np.newaxis]
                diff_NM = psin[j, :][np.newaxis, :] - u[jj, :][di_full][:, np.newaxis]
                KM  += beta[i, j] * diff_M**2
                KNM += beta[i, j] * diff_NM**2
            KM  = gamma[i] * np.exp(-KM)
            KNM = gamma[i] * np.exp(-KNM)

            KC  = cholesky(KM + (1/qtr[i]) * (KNM.T @ (d_i[:, None] * KNM)) + 1e-5 * np.eye(M))
            der = ((xs[i, di_full+1] - xs[i, di_full])
                   - d_i * (mb[i] - ma[i] * xs[i, di_full])) / qtr[i]
            ld  = np.linalg.solve(KC.T, KNM.T @ der)

            nrY = sum(
                np.sum((yhat[i, Ser[0,l]+1:Ser[1,l]+1] - yhat[i, Ser[0,l]:Ser[1,l]])**2
                       / (Tsam[l][1:] - Tsam[l][:-1]))
                for l in range(n_ser)
            )

            J1[i] = (0.5 * nrY / qtr[i]
                     - 0.5 * ld @ ld
                     + np.log(np.diag(KC)).sum()
                     - 0.5 * np.log(det(KM + 1e-5 * np.eye(M)))
                     - (mb[i] - ma[i] * xs[i, di_full]) @ (xs[i, di_full+1] - xs[i, di_full]) / qtr[i]
                     + 0.5 / qtr[i] * np.sum((d_i**0.5 * (mb[i] - ma[i] * xs[i, di_full]))**2))

        # Accept / reject trajectory proposal
        P_aux_q = np.exp(np.sum(
            1e-5 / q - 1e-5 / qtr
            + np.log(q/qtr) * (1.001 + 0.5 * (y.shape[1] - n_ser))
        ))
        if P_aux_q * np.exp(np.sum(Jold - J1)) > np.random.rand():
            Jold    = J1.copy()
            q       = qtr.copy()
            acctraj += 1
            xs_old  = xs.copy()
            yold    = yhat.copy()
            psiold  = psin.copy()

        # Early time warning
        if k == 100:
            elapsed = time.time() - t0
            left = elapsed * (parameters["its"] - k) / 100
            if left > 900:
                h_left   = int(left // 3600)
                min_left = int((left % 3600) // 60)
                print(f"NOTE! Estimated time remaining: {h_left} h {min_left} min")

        # Thinning: keep every 10th sample
        if k % 10 == 0:
            chain  += 1
            Plink  += Sold
            xstore += xs_old

            if k % 10000 == 0:
                elapsed  = time.time() - t0
                left     = elapsed * (parameters["its"] - k) / 10000
                h_left   = int(left // 3600)
                min_left = int((left % 3600) // 60)
                sec_left = int(left % 60)
                print(f"Iteration: {k}, Estimated time remaining: "
                      f"{h_left} h {min_left} min {sec_left} sec")
                time_mark = time.time()

    xstore /= (parameters["its"] / 10)

    stats["acctraj"] = acctraj
    stats["acctop"]  = acctop
    stats["acchyp"]  = acchyp
    stats["accr"]    = accr
    stats["fin_time"] = datetime.now()

    state["q"]     = q
    state["gamma"] = gamma
    state["r"]     = r
    state["xs"]    = xs_old
    state["P"]     = Pold
    state["bets"]  = betsold
    state["J"]     = Jold
    state["S"]     = Sold
    state["psi"]   = psiold
    state["ma"]    = ma
    state["mb"]    = mb

    return Plink, chain, xstore, state, stats


# ---------------------------------------------------------------------------
# Helper — must be implemented separately (mirrors match_check.m)
# ---------------------------------------------------------------------------
def match_check(a, b):
    """Return True if dimensions / lengths are inconsistent."""
    return len(a) != len(b)

In [2]:
import numpy as np

n, T, M = 3, 10, 5   # genes, timepoints, pseudoinputs

data = {
    "ts":   [np.random.randn(n, T)],
    "Tsam": [np.linspace(0, 9, T)],
}

state = {
    "q":     np.ones(n) * 0.1,
    "gamma": np.ones(n) * 1.0,
    "r":     np.ones(n) * 0.01,
    "xs":    np.random.randn(n, (T-1)*2 + 1),  # nstep=2 example
    "P":     np.zeros(n),
    "bets":  np.zeros((n, n)),
    "J":     np.zeros(n),
    "S":     np.zeros((n, n)),
    "psi":   np.random.randn(n, M),
    "ma":    np.ones(n) * 0.1,
    "mb":    np.zeros(n),
}

parameters = {
    "its": 1000, "ebeta": 0.2, "egamma": 0.1,
    "ea": 0.01,  "eb": 0.01,   "er": 0.01,
    "eq": 0.01,  "etraj": 0.2,
}

In [3]:
import numpy as np
 # assumes BINGO.py is in the same folder


# =============================================================================
# 1. SETTINGS
# =============================================================================
np.random.seed(42)

n      = 4    # number of genes
T      = 15   # timepoints per experiment
n_exp  = 2    # number of experiments
nstep  = 2    # interpolation factor (fine-grid steps between observations)
M      = 6    # number of GP pseudoinputs


# =============================================================================
# 2. BUILD DATA
# =============================================================================

# --- Time series: list of (n x T) arrays, one per experiment ---
ts = [np.random.randn(n, T) for _ in range(n_exp)]

# --- Sampling times: list of 1D arrays, one per experiment ---
Tsam = [np.linspace(0, T - 1, T) for _ in range(n_exp)]

data = {
    "ts":   ts,
    "Tsam": Tsam,
}

# --- Optional: knockout info ---
# Gene 2 is knocked out in experiment 1 (0-indexed)
# data["ko"] = [[], [2]]

# --- Optional: known/forbidden links as (n x n) matrix ---
# +1 = link must exist, -1 = link must not exist, 0 = unknown
# data["sure"] = np.zeros((n, n))

# --- Optional: external inputs ---
# Each array is (n_in x T), note: one extra column required
# data["input"] = [np.random.randn(1, T) for _ in range(n_exp)]

# --- Optional: missing measurements as list of (k x 2) arrays ---
# Each row = [gene_index, timepoint_index]
# data["missing"] = [np.empty((0, 2)), np.array([[1, 3], [2, 7]])]


# =============================================================================
# 3. INITIALISE STATE
# =============================================================================

# Fine-grid trajectory length:
#   Each experiment contributes (T-1)*nstep + 1 fine points,
#   but experiments are concatenated with shared endpoints removed.
#   Total = n_exp + (T*n_exp - n_exp) * nstep  ...simplified:
Nfine = n_exp + (T * n_exp - n_exp) * nstep

state = {
    # Process noise variance — one per gene, small positive value
    "q":     np.ones(n) * 0.05,

    # GP output scale — one per gene, start near signal variance
    "gamma": np.ones(n) * 1.0,

    # Measurement noise variance — one per gene
    "r":     np.ones(n) * 0.01,

    # Latent trajectory on the fine grid — (n x Nfine)
    # Good init: linear interpolation of observed data
    "xs":    np.random.randn(n, Nfine) * 0.1,

    # Log prior probability for current topology row — initialise to 0
    "P":     np.zeros(n),

    # Relevance parameters — (n x n), initialise to 0
    "bets":  np.zeros((n, n)),

    # Cost function values — one per gene, initialise to 0
    "J":     np.zeros(n),

    # Network topology (binary adjacency) — (n x n), start with no links
    "S":     np.zeros((n, n)),

    # Pseudoinput locations — (n x M), scatter within data range
    "psi":   np.random.randn(n, M),

    # Mean-reversion rate — small positive
    "ma":    np.ones(n) * 0.1,

    # Basal expression level — initialise near zero
    "mb":    np.zeros(n),
}


# =============================================================================
# 4. SET PARAMETERS
# =============================================================================
parameters = {
    "its":    5000,   # total MCMC iterations (use 50000+ for real runs)

    # Step sizes — tune so acceptance rates land roughly 20-40%
    "ebeta":  0.2,    # relevance parameters
    "egamma": 0.1,    # GP output scale
    "ea":     0.01,   # mean-reversion rate ma
    "eb":     0.01,   # basal expression mb
    "er":     0.01,   # measurement noise r
    "eq":     0.01,   # process noise q

    # Trajectory Crank-Nicolson step (0 < etraj < 1)
    # Smaller = more correlated proposals, higher acceptance
    "etraj":  0.2,

    # Topology temperature (default 1; increase to flatten topology posterior)
    "Theur":  1.0,
}


# =============================================================================
# 5. RUN
# =============================================================================
print("Starting BINGO...")
Plink, chain, xstore, state_out, stats = BINGO(data, state, parameters)


# =============================================================================
# 6. RESULTS
# =============================================================================

# Plink / chain = marginal posterior probability of each link
link_prob = Plink / chain
print("\n--- Inferred link probabilities (rows = target gene) ---")
print(np.round(link_prob, 3))

# Common threshold: keep links with probability > 0.5
network = (link_prob > 0.5).astype(int)
print("\n--- Inferred network (threshold 0.5) ---")
print(network)

print(f"\nChain length (thinned samples): {chain}")
print(f"Trajectory acceptance rate:     {stats['acctraj'] / parameters['its']:.3f}")
print(f"Topology acceptance rates:      {stats['acctop'] / parameters['its']}")
print(f"Hyperpar acceptance rates:      {stats['acchyp'] / parameters['its']}")
print(f"Noise var acceptance rates:     {stats['accr']   / parameters['its']}")

# Posterior mean trajectory (fine grid)
print(f"\nPosterior mean trajectory shape: {xstore.shape}")

Starting BINGO...


ValueError: operands could not be broadcast together with shapes (27,) (4,28) 

In [3]:

import numpy as np
import scipy.io

mat = scipy.io.loadmat("Example_Data.mat")

In [8]:
mat

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Thu Sep 19 14:47:05 2019',
 '__version__': '1.0',
 '__globals__': [],
 'X1': array([[ 3.42908546,  3.19274123,  3.53839328,  3.31847314,  3.69705432,
          2.82817993,  3.01008433,  2.92454082,  2.71791902,  3.18131067,
          2.88045811,  2.34866792,  2.73032284,  2.58692595,  2.46522401,
          2.98124499,  3.16088835,  3.58536949,  2.99753553,  2.24732407,
          4.41721861],
        [ 1.41638522,  3.0676972 ,  3.1863934 ,  2.86228189,  3.02973054,
          3.17748439,  4.07581682,  3.51979283,  2.70618893,  2.64616315,
          3.37179791,  4.10344852,  3.65476354,  3.7197787 ,  2.979342  ,
          3.25321735,  4.65458835,  4.62677458,  4.23685337,  4.27564477,
          4.04994982],
        [10.47417432,  5.64088488,  4.27369217,  3.17409743,  2.35662506,
          3.2554804 ,  3.41231532,  3.16116015,  2.62988054,  2.16487516,
          3.12122632,  3.36231078,  4.39396574,  3.32801989,  3.552582

In [ ]:
import numpy as np
import scipy.io



# =============================================================================
# 1. LOAD .mat FILE  —  update path if needed
# =============================================================================
mat = scipy.io.loadmat("Example_Data.mat", simplify_cells=True)
print("Variables found:", [k for k in mat.keys() if not k.startswith("__")])

# Pull out the four experiments and sampling info
X1 = mat["X1"].astype(float)   # 5 x 21
X2 = mat["X2"].astype(float)   # 5 x 14
X3 = mat["X3"].astype(float)   # 5 x 21
X4 = mat["X4"].astype(float)   # 5 x 14
dt1   = float(mat["dt1"])                     # 0.5
times = mat["times"].ravel().astype(float)    # 1 x 14  → the short time vector

ts   = [X1, X2, X3, X4]
Tsam = [
    np.arange(X1.shape[1]) * dt1,   # 0, 0.5, ..., 10.0  (21 pts)
    times,                            # provided directly   (14 pts)
    np.arange(X3.shape[1]) * dt1,
    times,
]

data = {"ts": ts, "Tsam": Tsam}


# =============================================================================
# 2. BINGO_init  (Python translation)
# =============================================================================
def BINGO_init(data):
    nstep  = 4    # fine-grid steps between measurements
    nr_pi  = 50   # number of pseudoinputs

    ts   = data["ts"]
    Tsam = data["Tsam"]
    n    = ts[0].shape[0]
    n_ser = len(ts)

    # ---- parameters --------------------------------------------------------
    parameters = {
        "etraj":  0.1,
        "egamma": 0.1,
        "ea":     0.005,
        "eb":     0.005,
        "ebeta":  0.125,
        "er":     0.0001,
        "eq":     0.0002,
        "Theur":  1,
        "its":    3000,
    }

    # ---- scale data so each gene spans [0, 1] across all experiments -------
    maxs = np.full(n, -np.inf)
    mins = np.full(n,  np.inf)
    for y in ts:
        maxs = np.maximum(maxs, y.max(axis=1))
        mins = np.minimum(mins, y.min(axis=1))

    scale = maxs - mins                        # shape (n,)
    ts_scaled = [y / scale[:, None] for y in ts]
    data["ts"] = ts_scaled

    maxs_s = maxs / scale
    mins_s = maxs_s - 1.0

    # ---- Ser: index bookkeeping (0-based) ----------------------------------
    #   Ser[0,j], Ser[1,j]  : start/end columns in the concatenated y matrix
    #   Ser[2,j], Ser[3,j]  : start/end columns in the fine-grid xs matrix
    Ser = np.zeros((4, n_ser), dtype=int)
    Ser[0, 0] = 0
    Ser[1, 0] = ts_scaled[0].shape[1] - 1
    Ser[2, 0] = 0
    Ser[3, 0] = nstep * (Ser[1, 0] - Ser[0, 0])
    for j in range(1, n_ser):
        Ser[0, j] = Ser[1, j-1] + 1
        Ser[1, j] = Ser[1, j-1] + ts_scaled[j].shape[1]
        Ser[2, j] = Ser[3, j-1] + 1
        Ser[3, j] = Ser[3, j-1] + nstep * (Ser[1, j] - Ser[0, j])

    Nfine = int(Ser[3, -1]) + 1

    # ---- initial trajectory: piecewise-linear interpolation ----------------
    xs = np.zeros((n, Nfine))
    for j in range(n_ser):
        y   = ts_scaled[j]
        T_j = y.shape[1]
        sf  = Ser[2, j]
        for jj in range(T_j - 1):
            col0 = sf + jj * nstep
            for s in range(nstep):
                alpha = s / nstep
                xs[:, col0 + s] = (1 - alpha) * y[:, jj] + alpha * y[:, jj+1]
        xs[:, Ser[3, j]] = y[:, -1]

    # ---- signal variance estimates for hyperparameter init -----------------
    nry   = np.zeros(n)
    nry_q = np.zeros(n)
    Ttot  = np.zeros(n)
    for j in range(n_ser):
        y  = ts_scaled[j]
        dt = Tsam[j][1:] - Tsam[j][:-1]
        nry   += np.sum((y[:, 1:] - y[:, :-1])**2 / dt, axis=1)
        nry_q += np.sum((y[:, 1:] - y[:, :-1])**2,      axis=1)
        Ttot  += Tsam[j][-1] - Tsam[j][0]
    nry   /= Ttot
    nry_q /= Ttot

    # ---- pseudoinputs: uniform random within scaled data range -------------
    psi = (mins_s[:, None]
           + (maxs_s - mins_s)[:, None] * np.random.rand(n, nr_pi))

    # ---- state -------------------------------------------------------------
    state = {
        "S":     (np.random.rand(n, n) > 0.9).astype(float),
        "bets":  np.abs(np.random.randn(n, n) * 0.5) + 1e-3,
        "psi":   psi,
        "xs":    xs,
        "r":     np.full(n, 0.0006 * 0.1),
        "gamma": nry.copy(),
        "q":     nry_q / 20,
        "ma":    np.full(n, 0.1),
        "mb":    np.full(n, 0.05),
        "P":     np.full(n, -1e8),
        "J":     np.full(n,  1e8),
    }

    return data, state, parameters


# =============================================================================
# 3. INITIALISE
# =============================================================================
data, state, parameters = BINGO_init(data)
print(f"\nData scaled. Fine-grid trajectory shape: {state['xs'].shape}")


# =============================================================================
# 4. BURN-IN
# =============================================================================
print("\n--- BURN-IN ---")
_, chain, _, state, stats = BINGO(data, state, parameters)
print(f"Burn-in complete. Traj acceptance: {stats['acctraj']/parameters['its']:.3f}")


# =============================================================================
# 5. SAMPLING  (first pass)
# =============================================================================
parameters["its"] = 10000
print("\n--- SAMPLING (pass 1) ---")
Plink, chain, xstore, state, stats = BINGO(data, state, parameters)
print(f"Sampling complete. Chain length: {chain}")
confidence_matrix = Plink / chain


# =============================================================================
# 6. COLLECT MORE SAMPLES  (optional second pass, mirrors the MATLAB script)
# =============================================================================
chain_old  = chain
Plink_old  = Plink.copy()
xstore_old = xstore.copy()

parameters["its"] = 10000
print("\n--- SAMPLING (pass 2) ---")
Plink, chain, xstore, state, stats = BINGO(data, state, parameters)

# Combine both passes
xstore            = (chain_old / (chain + chain_old)) * xstore_old \
                  + (chain     / (chain + chain_old)) * xstore
Plink             = Plink_old + Plink
chain             = chain_old + chain
confidence_matrix = Plink / chain

print(f"Combined chain length: {chain}")


# =============================================================================
# 7. RESULTS
# =============================================================================
network = (confidence_matrix > 0.5).astype(int)

print("\n--- Confidence matrix (link probabilities) ---")
print(np.round(confidence_matrix, 3))
print("\n--- Inferred network (threshold 0.5) ---")
print(network)

np.savetxt("confidence_matrix.csv", confidence_matrix, delimiter=",", fmt="%.4f")
np.savetxt("network.csv",           network,           delimiter=",", fmt="%d")
print("\nSaved: confidence_matrix.csv, network.csv")

Variables found: ['X1', 'X2', 'X3', 'X4', 'dt1', 'times']

Data scaled. Fine-grid trajectory shape: (5, 265)

--- BURN-IN ---


766.71s - invalid syntax. Perhaps you forgot a comma? (<string>, line 1)
Traceback (most recent call last):
  File "/home/dbiparva/anaconda3/envs/gpytorch/lib/python3.14/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/pydevd_vars.py", line 636, in change_attr_expression
    value = eval(expression, frame.f_globals, frame.f_locals)
  File "<string>", line 1
    {'ts': [array([[0.47560387, 0.44282363, 0.49076453, 0.46026227, 0.51277035,        0.3922600...6545029, 0.35919078,        0.32310773]]), array([[0.46475698, 0.58210578, 0.80921915, 0.70084626, 0.67486228,        0.6091821...961, 0.57319431, 0.58726837, 0.55998709]]), array([[0.34467713, 0.49652152, 0.56636064, 0.69689019, 0.67079758,        0.5817379...6938938, 0.59236461,        0.46557836]]), array([[ 0.47741069,  0.20332166,  0.17800804,  0.04243468,  0.04747989,         0.0...,  0.19866173,  0.12974163,  0.22204036]])], 'Tsam': [array([ 0. ,  0.5,  1. ,  1.5,  2. ,  2.5,  3. ,  3.5,  4. ,  4.5,  5. ,        5.5,... 7. 

In [1]:
mat = scipy.io.loadmat("Example_Data.mat", simplify_cells=True)
print("Variables found:", [k for k in mat.keys() if not k.startswith("__")])

NameError: name 'scipy' is not defined